<a href="https://colab.research.google.com/github/SmritiD2004/RAG--IBMSkillsbuild/blob/main/Dynamic_RAG_Web_Crawlingipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-generativeai beautifulsoup4 requests

In [ ]:
import faiss
import numpy as np
import requests
import re
import google.generativeai as genai

from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from google.colab import userdata

In [ ]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini initialized successfully!")

In [ ]:
user_query = input("Enter your query: ")

response = model.generate_content(user_query)

baseline_response = response.text

print("\n===== Baseline Response (Without RAG) =====\n")
print(baseline_response)

In [ ]:
url = "https://thealliance.ai/"

try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove unwanted HTML tags
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    page_text = soup.get_text(separator="\n")

    page_text = re.sub(r"\n\s*\n+", "\n\n", page_text)

    with open("thealliance_ai_content.txt", "w", encoding="utf-8") as f:
        f.write(page_text)

    print("Website crawled successfully!")
    print(f"Characters extracted: {len(page_text)}")

except Exception as e:
    print("Web crawling failed:", e)

In [ ]:
def split_into_chunks(text, chunk_size=300, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = split_into_chunks(page_text)

print(f"Created {len(chunks)} chunks.")

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(
    chunks,
    convert_to_numpy=True
).astype(np.float32)

print("Embeddings Shape:", embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

chunk_store = chunks

print(f"Indexed {index.ntotal} document chunks.")

In [ ]:
query_embedding = embed_model.encode(
    [user_query],
    convert_to_numpy=True
).astype(np.float32)

k = 3

distances, indices = index.search(query_embedding, k)

retrieved_chunks = [chunk_store[i] for i in indices[0]]

print("Top Retrieved Chunks:\n")

for i, chunk in enumerate(retrieved_chunks, start=1):
    print("=" * 60)
    print(f"Chunk {i}")
    print("=" * 60)
    print(chunk)
    print()

In [ ]:
retrieved_context = "\n\n".join(retrieved_chunks)

prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer is not available in the context, reply:

"I don't have enough information in the provided context."

Context:
{retrieved_context}

Question:
{user_query}
"""

In [ ]:
response = model.generate_content(prompt)

rag_response = response.text

print("\n===== RAG Response =====\n")

print(rag_response)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"""
# User Query

**{user_query}**

---

# Baseline Response (Without RAG)

{baseline_response}

---

# Retrieved Context

{retrieved_context}

---

# RAG Response

{rag_response}
"""))